In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Kaggle dataset path
data_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset"

for split in ['training', 'validation', 'test']:
    split_path = os.path.join(data_dir, split)
    print(f"\nContents of {split} folder:")

    total_images = 0

    if split != 'test':
        # Count images inside class subfolders
        for class_name in os.listdir(split_path):
            class_path = os.path.join(split_path, class_name)
            if os.path.isdir(class_path):
                num_images = len([
                    f for f in os.listdir(class_path)
                    if os.path.isfile(os.path.join(class_path, f))
                ])
                total_images += num_images
                print(f"  {class_name}: {num_images} images")
    else:
        # Test folder has images directly
        num_images = len([
            f for f in os.listdir(split_path)
            if os.path.isfile(os.path.join(split_path, f))
        ])
        total_images += num_images
        print(f"  (no subfolders): {num_images} images")

    print(f"Total images in {split}: {total_images}")


In [ ]:
IMG_SIZE = (128,128)
BATCH_SIZE = 32

train_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/training"
val_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/validation"
test_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/test"

In [ ]:
import os, warnings, logging
import tensorflow as tf

# Suppress TensorFlow C++ backend logs (INFO, WARNING, ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"   # optional: disables some verbose oneDNN logs
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"
warnings.filterwarnings("ignore")
tf.get_logger().setLevel(logging.ERROR)

# Configure GPU memory growth (avoids allocator warnings)
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)



# Version 1

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Flatten, Dense, Dropout

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

In [ ]:

V1model = Sequential([
        Input(shape=(128,128,3)),

        Conv2D(32,(3,3), activation='relu'),
        MaxPool2D(2,2),

        Conv2D(64,(3,3), activation='relu'),
        MaxPool2D(2,2),

        Conv2D(128,(3,3), activation='relu'),
        MaxPool2D(2,2),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(7, activation='softmax')
    ])
    

In [ ]:
V1model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
V1model.summary()

In [ ]:
historyV1 = V1model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy plot
axs[0].plot(historyV1.history['accuracy'], label='train acc')
axs[0].plot(historyV1.history['val_accuracy'], label='val acc')
axs[0].set_title("Training vs Validation Accuracy")
axs[0].legend()

# Loss plot
axs[1].plot(historyV1.history['loss'], label='train loss')
axs[1].plot(historyV1.history['val_loss'], label='val loss')
axs[1].set_title("Training vs Validation Loss")
axs[1].legend()

plt.tight_layout()
plt.savefig("V1_accuracy_loss.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
V1model.save("Currency_v1.keras")

## Load the model

In [ ]:
from tensorflow import keras

model_v1 = keras.models.load_model("Currency_v1.keras")
print("Model loaded successfully!")


In [ ]:
# Map class indices to denominations
class_labels = [10, 20, 50, 100, 200, 500, 2000]

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

val_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/validation"

val_datagen = ImageDataGenerator(rescale=1./255)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
from tensorflow import keras
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# denomination labels 
denominations = [10, 20, 50, 100, 200, 500, 2000]

# Predict on validation data
y_probs = model_v1.predict(val_data)   # probabilities
y_pred = np.argmax(y_probs, axis=1)    # predicted indices
y_true = val_data.classes              # true indices

# Classification report with denominations
print(classification_report(
    y_true,
    y_pred,
    target_names=[f"₹{lbl}" for lbl in denominations]
))

# Confusion matrix with denominations
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

# Accuracy
acc = accuracy_score(y_true, y_pred)
print("Validation Accuracy:", acc)


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Binarize true labels for multi-class ROC
y_true_bin = label_binarize(y_true, classes=range(len(denominations)))

# Macro ROC–AUC
roc_auc = roc_auc_score(y_true_bin, y_probs, average="macro", multi_class="ovr")
print("Macro ROC–AUC:", roc_auc)

# Plot ROC curves per denomination
plt.figure(figsize=(8,6))
for i, lbl in enumerate(denominations):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc_class = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"₹{lbl} (AUC = {roc_auc_class:.2f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (Validation Set)")
plt.legend(loc="lower right")
plt.savefig("roc_curves.png", dpi=300, bbox_inches='tight')
plt.show()


## Predicting random notes from test data with a threshold 0.75

In [ ]:
import random
from tensorflow.keras.preprocessing import image

test_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/test"

# Mapping index → denomination
index_to_label = {
    0: "10",
    1: "20",
    2: "50",
    3: "100",
    4: "200",
    5: "500",
    6: "2000"
}

CONF_THRESHOLD = 0.75

# Pick 5 random images
sample_images = random.sample(os.listdir(test_dir), 5)

for img_name in sample_images:
    img_path = os.path.join(test_dir, img_name)

    # Load and preprocess
    img = image.load_img(img_path, target_size=(128 , 128))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    pred_probs = model_v1.predict(img_array)
    pred_class = np.argmax(pred_probs, axis=1)[0]
    pred_conf  = np.max(pred_probs)

    # Decide label
    if pred_conf < CONF_THRESHOLD:
        pred_label = "Not a note"
    else:
        pred_label = index_to_label[pred_class]

    # Show image + prediction
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{img_name} → {pred_label} (conf={pred_conf:.2f})")
    plt.show()

# Version 2

In [ ]:
train_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/training"
val_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/validation"
test_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/test"

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

In [ ]:
IMG_SIZE = (128,128)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

## base Model - EfficientNetB0

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

## freeze the model


In [ ]:

for layer in base_model.layers:
    layer.trainable = False

## Add Custom Classification Head

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.4)(x)

output = layers.Dense(7, activation='softmax')(x)

V2model = models.Model(inputs=base_model.input, outputs=output)

In [ ]:
V2model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
V2model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint("Currency_v2.keras", save_best_only=True)
]

In [ ]:
history_v2 = V2model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy plot
axs[0].plot(history_v2.history['accuracy'], label='train acc')
axs[0].plot(history_v2.history['val_accuracy'], label='val acc')
axs[0].set_title("Training vs Validation Accuracy")
axs[0].legend()

# Loss plot
axs[1].plot(history_v2.history['loss'], label='train loss')
axs[1].plot(history_v2.history['val_loss'], label='val loss')
axs[1].set_title("Training vs Validation Loss")
axs[1].legend()

plt.tight_layout()
plt.savefig("V2_accuracy_loss.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
V2model.save("Currency_v2.keras")

## Load v2 model

In [ ]:
from tensorflow import keras

model_v2 = keras.models.load_model("Currency_v2.keras")
print("Model loaded successfully!")

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

val_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/validation"

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
from tensorflow import keras
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# denomination labels 
denominations = [10, 20, 50, 100, 200, 500, 2000]

# Predict on validation data
y_probs = model_v2.predict(val_data)   # probabilities
y_pred = np.argmax(y_probs, axis=1)    # predicted indices
y_true = val_data.classes              # true indices

# Classification report with denominations
print(classification_report(
    y_true,
    y_pred,
    target_names=[f"₹{lbl}" for lbl in denominations]
))

# Confusion matrix with denominations
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

# Accuracy
acc = accuracy_score(y_true, y_pred)
print("Validation Accuracy:", acc)


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Binarize true labels for multi-class ROC
y_true_bin = label_binarize(y_true, classes=range(len(denominations)))

# Macro ROC–AUC
roc_auc = roc_auc_score(y_true_bin, y_probs, average="macro", multi_class="ovr")
print("Macro ROC–AUC:", roc_auc)

# Plot ROC curves per denomination
plt.figure(figsize=(8,6))
for i, lbl in enumerate(denominations):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc_class = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"₹{lbl} (AUC = {roc_auc_class:.2f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (Validation Set)")
plt.legend(loc="lower right")
plt.savefig("roc_curves_v2.png", dpi=300, bbox_inches='tight')
plt.show()


## Predicting random notes from test data with a threshold 0.75

In [ ]:
import random
from tensorflow.keras.preprocessing import image

test_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/test"

# Mapping index → denomination
index_to_label = {
    0: "10",
    1: "20",
    2: "50",
    3: "100",
    4: "200",
    5: "500",
    6: "2000"
}

CONF_THRESHOLD = 0.75

# Pick 5 random images
sample_images = random.sample(os.listdir(test_dir), 5)

for img_name in sample_images:
    img_path = os.path.join(test_dir, img_name)

    # Load and preprocess
    img = image.load_img(img_path, target_size=(128 , 128))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    pred_probs = model_v2.predict(img_array)
    pred_class = np.argmax(pred_probs, axis=1)[0]
    pred_conf  = np.max(pred_probs)

    # Decide label
    if pred_conf < CONF_THRESHOLD:
        pred_label = "Not a note"
    else:
        pred_label = index_to_label[pred_class]

    # Show image + prediction
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{img_name} → {pred_label} (conf={pred_conf:.2f})")
    plt.show()

# Version 3

- Partial Fine-Tuning + Data Augmentation (EfficientNetB0)

In [ ]:
train_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/training"
val_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/validation"
test_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/test"

In [ ]:
IMG_SIZE = (128,128)
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 7

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

# Data Augmentation with preprocess_input

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

## Base model - EfficentNetB0

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

In [ ]:
## Partial fine-tuning

for layer in base_model.layers[:-30]:  # freeze most
    layer.trainable = False

for layer in base_model.layers[-30:]:  # unfreeze top layers
    layer.trainable = True

In [ ]:
## Add Custom Head

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

V3model = models.Model(inputs=base_model.input, outputs=outputs)

In [ ]:
V3model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
V3model.summary()

In [ ]:
## Early stoppings

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=4, restore_best_weights=True),
    
    ModelCheckpoint("Currency_v3.keras", save_best_only=True),
    
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

In [ ]:
history_v3 = V3model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
val_loss, val_acc = V3model.evaluate(val_data)

print("Validation Accuracy:", val_acc)
print("Validation Loss:", val_loss)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy plot
axs[0].plot(history_v3.history['accuracy'], label='train acc')
axs[0].plot(history_v3.history['val_accuracy'], label='val acc')
axs[0].set_title("Training vs Validation Accuracy")
axs[0].legend()

# Loss plot
axs[1].plot(history_v3.history['loss'], label='train loss')
axs[1].plot(history_v3.history['val_loss'], label='val loss')
axs[1].set_title("V3 Training vs Validation Loss")
axs[1].legend()

plt.tight_layout()
plt.savefig("V3_accuracy_loss.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
V3model.save("Currency_v3.keras")

In [ ]:
from tensorflow import keras

model_v3 = keras.models.load_model("Currency_v3.keras")
print("Model loaded successfully!")

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

val_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/validation"

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
from tensorflow import keras
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# denomination labels 
denominations = [10, 20, 50, 100, 200, 500, 2000]

# Predict on validation data
y_probs = model_v3.predict(val_data)   # probabilities
y_pred = np.argmax(y_probs, axis=1)    # predicted indices
y_true = val_data.classes              # true indices

# Classification report with denominations
print(classification_report(
    y_true,
    y_pred,
    target_names=[f"₹{lbl}" for lbl in denominations]
))

# Confusion matrix with denominations
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

# Accuracy
acc = accuracy_score(y_true, y_pred)
print("Validation Accuracy:", acc)


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Binarize true labels for multi-class ROC
y_true_bin = label_binarize(y_true, classes=range(len(denominations)))

# Macro ROC–AUC
roc_auc = roc_auc_score(y_true_bin, y_probs, average="macro", multi_class="ovr")
print("Macro ROC–AUC:", roc_auc)

# Plot ROC curves per denomination
plt.figure(figsize=(8,6))
for i, lbl in enumerate(denominations):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc_class = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"₹{lbl} (AUC = {roc_auc_class:.2f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (Validation Set)")
plt.legend(loc="lower right")
plt.savefig("roc_curves_v3.png", dpi=300, bbox_inches='tight')
plt.show()


## ## Predicting random notes from test data with a threshold 0.75

In [ ]:
import random
from tensorflow.keras.preprocessing import image

test_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/test"

# Mapping index → denomination
index_to_label = {
    0: "10",
    1: "20",
    2: "50",
    3: "100",
    4: "200",
    5: "500",
    6: "2000"
}

CONF_THRESHOLD = 0.75

# Pick 5 random images
sample_images = random.sample(os.listdir(test_dir), 5)

for img_name in sample_images:
    img_path = os.path.join(test_dir, img_name)

    # Load and preprocess
    img = image.load_img(img_path, target_size=(128 , 128))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    pred_probs = model_v3.predict(img_array)
    pred_class = np.argmax(pred_probs, axis=1)[0]
    pred_conf  = np.max(pred_probs)

    # Decide label
    if pred_conf < CONF_THRESHOLD:
        pred_label = "Not a note"
    else:
        pred_label = index_to_label[pred_class]

    # Show image + prediction
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{img_name} → {pred_label} (conf={pred_conf:.2f})")
    plt.show()

# Version 4 

- **Full fine-tuning (all layers trainable)**

In [ ]:
train_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/training"
val_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/validation"
test_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/test"

In [ ]:
IMG_SIZE = (128,128)
BATCH_SIZE = 32
EPOCHS = 10

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

## Load V3 model

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
from keras.models import load_model

V4model = load_model("/kaggle/working/Currency_v3.keras")


In [ ]:
# Unfreeze all layers

for layer in V4model.layers: 
    layer.trainable = True

In [ ]:
# Recompile with VERY LOW Learning Rate

V4model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
V4model.summary()

In [ ]:
# Callbacks

callbacks = [
    EarlyStopping(patience=4, restore_best_weights=True),

    ModelCheckpoint("Currency_v4.keras", save_best_only=True),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

In [ ]:
history_v4 = V4model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
val_loss, val_acc = V4model.evaluate(val_data)

print("Final Validation Accuracy:", val_acc)
print("Final Validation Loss:", val_loss)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy plot
axs[0].plot(history_v4.history['accuracy'], label='train acc')
axs[0].plot(history_v4.history['val_accuracy'], label='val acc')
axs[0].set_title("Training vs Validation Accuracy")
axs[0].legend()

# Loss plot
axs[1].plot(history_v4.history['loss'], label='train loss')
axs[1].plot(history_v4.history['val_loss'], label='val loss')
axs[1].set_title("V4 Training vs Validation Loss")
axs[1].legend()

plt.tight_layout()
plt.savefig("V4_accuracy_loss.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
V4model.save("Currency_v4.keras")

In [ ]:
from tensorflow import keras

model_v4 = keras.models.load_model("Currency_v4.keras")
print("Model loaded successfully!")

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

val_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/validation"

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
from tensorflow import keras
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# denomination labels 
denominations = [10, 20, 50, 100, 200, 500, 2000]

# Predict on validation data
y_probs = model_v4.predict(val_data)   # probabilities
y_pred = np.argmax(y_probs, axis=1)    # predicted indices
y_true = val_data.classes              # true indices

# Classification report with denominations
print(classification_report(
    y_true,
    y_pred,
    target_names=[f"₹{lbl}" for lbl in denominations]
))

# Confusion matrix with denominations
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

# Accuracy
acc = accuracy_score(y_true, y_pred)
print("Validation Accuracy:", acc)


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Binarize true labels for multi-class ROC
y_true_bin = label_binarize(y_true, classes=range(len(denominations)))

# Macro ROC–AUC
roc_auc = roc_auc_score(y_true_bin, y_probs, average="macro", multi_class="ovr")
print("Macro ROC–AUC:", roc_auc)

# Plot ROC curves per denomination
plt.figure(figsize=(8,6))
for i, lbl in enumerate(denominations):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc_class = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"₹{lbl} (AUC = {roc_auc_class:.2f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (Validation Set)")
plt.legend(loc="lower right")
plt.savefig("roc_curves_v4.png", dpi=300, bbox_inches='tight')
plt.show()


## Predicting random notes from test data with a threshold 0.75

In [ ]:
import random
from tensorflow.keras.preprocessing import image

test_dir = "/kaggle/input/datasets/omkarshinde23/indian-currency-notes/dataset/test"

# Mapping index → denomination
index_to_label = {
    0: "10",
    1: "20",
    2: "50",
    3: "100",
    4: "200",
    5: "500",
    6: "2000"
}

CONF_THRESHOLD = 0.75

# Pick 5 random images
sample_images = random.sample(os.listdir(test_dir), 5)

for img_name in sample_images:
    img_path = os.path.join(test_dir, img_name)

    # Load and preprocess
    img = image.load_img(img_path, target_size=(128 , 128))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    pred_probs = model_v4.predict(img_array)
    pred_class = np.argmax(pred_probs, axis=1)[0]
    pred_conf  = np.max(pred_probs)

    # Decide label
    if pred_conf < CONF_THRESHOLD:
        pred_label = "Not a note"
    else:
        pred_label = index_to_label[pred_class]

    # Show image + prediction
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{img_name} → {pred_label} (conf={pred_conf:.2f})")
    plt.show()